# Классификатор тональности русских твитов: TF-IDF + MLP

**Задача:** бинарная классификация твитов на позитивные и негативные на датасете RuTweetCorp.

**Подход:** классический NLP-пайплайн: очистка и лемматизация текста → векторизация TF-IDF → многослойный перцептрон (MLP).
Дополнительно разбирается, на какие слова опирается модель, и где у такого подхода заканчивается «понимание» текста.

**Содержание**
1. Загрузка и предобработка данных
2. Векторизация TF-IDF и обучение MLP
3. Интерпретация модели: анализ весов
4. Оценка качества и слабые места
5. Тот же MLP на PyTorch
6. Отчёт и выводы

**Запуск:** Google Colab (`Runtime → Run all`) или локально через Docker (`docker compose up -d`, см. README).

## 0. Подготовка окружения

В Colab большинство библиотек уже установлено, не хватает только морфологического анализатора `pymorphy3` для лемматизации.
В Docker-образе всё ставится из `requirements.txt`, поэтому установка выполняется только в Colab.

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pymorphy3 pymorphy3-dicts-ru

Фиксируем `seed` (зерно генератора случайных чисел): разбиение на выборки, перемешивание и начальные веса нейросети
будут одинаковыми при каждом запуске, и результаты можно воспроизвести.

In [2]:
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option('display.max_colwidth', 120)

## 1. Загрузка и предобработка данных

### 1.1 Загрузка датасета

[RuTweetCorp на Kaggle](https://www.kaggle.com/datasets/maximsuvorov/rutweetcorp) содержит два размеченных файла:
`positive.csv` и `negative.csv`. Текст твита лежит в колонке `ttext`.

Файлы скачиваются по прямой ссылке без аккаунта Kaggle и сохраняются в папку `data/`. При повторном запуске берутся с диска.

Метки классов: **позитив = 1**, **негатив = 0**. Модели бинарной классификации ожидают классы 0 и 1,
а `predict_proba` возвращает вероятность класса 1, то есть «вероятность позитива».

In [3]:
DATA_DIR = Path('../data') if Path('../data').is_dir() else Path('data')
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_URL = 'https://www.kaggle.com/api/v1/datasets/download/maximsuvorov/rutweetcorp/{name}'

DATASET_FILES = {'positive.csv': 1, 'negative.csv': 0}
LABEL_NAMES = {0: 'негатив', 1: 'позитив'}


def download_dataset_file(name: str) -> Path:
    csv_path = DATA_DIR / name
    if csv_path.exists():
        print(f'{name}: уже скачан')
        return csv_path

    print(f'{name}: скачиваю...')
    response = requests.get(KAGGLE_URL.format(name=name), timeout=300)
    response.raise_for_status()

    zip_path = DATA_DIR / f'{name}.zip'
    zip_path.write_bytes(response.content)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extract(name, DATA_DIR)
    zip_path.unlink()

    return csv_path


frames = []
for name, label in DATASET_FILES.items():
    part = pd.read_csv(download_dataset_file(name), usecols=['ttext'])
    part['label'] = label
    frames.append(part)

df = (
    pd.concat(frames, ignore_index=True)
    .rename(columns={'ttext': 'text'})
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f'Размер датасета: {df.shape[0]:,} твитов, {df.shape[1]} колонки')
df.head(10)

positive.csv: уже скачан


negative.csv: уже скачан


Размер датасета: 226,834 твитов, 2 колонки


,text,label
0,мыс на меня обиделась:(\r\nя ей даже ничего не сделала:(,0
1,"аааааааааааааааааааа,не хочу на работу :(",0
2,"У меня какой-то особенный вид ушей! :D, некоторые вакуумные наушники в моих ушах просто не держатся!",1
3,"@simonovkon он неплохой человек в жизни. Я работала в шоу-бизе, со многими знакома. Встречаются очень хорошие люди....",0
4,"RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни, отдыхали.\nКричали на улице:)\nМы настоящяя семья)",1
5,@passkrinDObroTA это ужасно:( так страашно,0
6,"RT @miwisakovab: Не грустим за летом, скоро будет следующее)",1
7,"Лежим с молой в постели , помогаем ему выбирать машину !)",1
8,"Я ВАС НАЙДУ, ОТРЕЖУ ПАЛЬЦЫ И ЗАСТАВЛЮ СОЖРАТЬ, ЕСЛИ ЕЩЕ РАЗ ДОБАВИТЕ МЕНЯ В ОБЩУЮ БЕСЕДУ ВКОНТАКТЕ С КАКОЙ-ТО ЛЕВОЙ ...",1
9,"@larkagerasimova Контоев Артем) рост, вес и тд интересует ?)",1


### 1.2 Первичный осмотр данных

Проверяем баланс классов, пропуски, дубликаты и длину твитов.

In [4]:
balance = pd.DataFrame({
    'твитов': df['label'].value_counts(),
    'доля': df['label'].value_counts(normalize=True).round(3),
}).rename(index=LABEL_NAMES)

print('Пустых текстов:', df['text'].isna().sum())
print('Дубликатов текста:', df['text'].duplicated().sum())
balance

Пустых текстов: 0
Дубликатов текста: 9394


,твитов,доля
label,,
позитив,114911,0.507
негатив,111923,0.493


In [5]:
words = df['text'].str.split().str.len()

words.groupby(df['label'].map(LABEL_NAMES)).describe().round(1)

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
негатив,111923.0,12.0,5.0,1.0,8.0,11.0,15.0,37.0
позитив,114911.0,12.5,4.7,1.0,9.0,12.0,16.0,40.0


### 1.3 Как размечен корпус: смайлики

Корпус размечали автоматически: твиты со «весёлыми» смайликами (`:)`, `)))`, `:D`) попали в позитив,
с «грустными» (`:(`, `(((`) - в негатив. Смайлики остались в текстах.

Посчитаем, в какой доле твитов каждого класса есть скобки:

In [6]:
brackets = pd.DataFrame({
    'есть ")"': df['text'].str.contains(')', regex=False),
    'есть "("': df['text'].str.contains('(', regex=False),
}).groupby(df['label'].map(LABEL_NAMES)).mean().round(3)

brackets

,"есть "")""","есть ""("""
label,,
негатив,0.000,0.947
позитив,0.829,0.000


**Что видно из таблицы:**
- в позитивных твитах `)` встречается в 82.9% случаев, а `(` не встречается ни разу;
- в негативных твитах `(` встречается в 94.7% случаев, а `)` не встречается ни разу.

Скобка почти однозначно выдаёт класс. Если её оставить, модель выучит правило «`)` → позитив, `(` → негатив»,
покажет высокую точность и при этом ничего не узнает о словах. Это **утечка данных (data leakage)**:
признак, по которому ставили метку, попал во входные данные. Поэтому при очистке текста (раздел 1.4)
скобки и смайлики удаляются обязательно.

**Другие наблюдения:**
- классы почти сбалансированы (50.7% / 49.3%), поэтому Accuracy будет честной метрикой;
- в среднем 12 слов на твит, длина у классов почти одинаковая;
- 9 394 текста повторяются (в основном ретвиты), с ними разберёмся после очистки;
- разметка шумная: она отражает смайлик, а не смысл. Например, угроза капслоком в строке 8 размечена как позитив
  (в ней был смайлик), а «он неплохой человек» в строке 3 — как негатив.

### 1.4 Очистка текста

TF-IDF считает слова «как есть»: `Привет`, `привет` и `ПРИВЕТ` для него три разных слова, а ссылки и никнеймы
только раздувают словарь. Функция `clean_text` по очереди:

| Шаг | Пример | Зачем |
|---|---|---|
| нижний регистр, `ё` → `е` | `ЁЛКА` → `елка` | одно слово — один признак |
| удаление ссылок | `http://t.co/GqG6iuE2` | каждая ссылка уникальна, тональности не несёт |
| удаление упоминаний | `@Darina_Lo` | имена пользователей — шум |
| удаление хештегов целиком | `#пятница` | по условию задания |
| всё, кроме русских букв, → пробел | `:)))`, `!!!`, `2`, `RT`, `:D` | пунктуация, цифры, латиница и **смайлики (утечка метки)** |
| 3+ одинаковые буквы подряд → одна | `домааааа` → `дома` | растянутые слова сводим к нормальным |
| смех в любом написании → `хаха` | `ахахах`, `хахаха`, `ахахха` | ~500 вариантов написания одного и того же сигнала; pymorphy не знает этих «слов» и разбирает их наугад (`ахахаха` → `ахахи`) |
| лишние пробелы и переносы строк → один пробел | `\n`, `   ` | аккуратный текст для токенизации |

Ссылки, упоминания и хештеги удаляются **до** фильтра букв: в них бывает кириллица (`@маша`, `#пятница`),
и фильтр оставил бы от них слово.

Вместо перечисления всего, что нужно удалить, используется «белый список»: оставляем только `а-я`.
Так в тексте гарантированно не останется ни одной скобки, эмодзи или латинского `:D`.

In [7]:
import re

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#\w+')
NON_CYRILLIC_RE = re.compile(r'[^а-я]+')
REPEATED_CHAR_RE = re.compile(r'(.)\1{2,}')
LAUGH_RE = re.compile(r'\b(?=[ах]*х)(?=[ах]*а)[ах]{3,}\b')


def clean_text(text: str) -> str:
    text = text.lower().replace('ё', 'е')
    text = URL_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)
    text = HASHTAG_RE.sub(' ', text)
    text = NON_CYRILLIC_RE.sub(' ', text)
    text = REPEATED_CHAR_RE.sub(r'\1', text)
    text = LAUGH_RE.sub('хаха', text)
    return ' '.join(text.split())


examples = [
    'RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни :))) ахахаххаха',
    'Смотри http://t.co/GqG6iuE2 #пятница, какая ЁЛКА!!! :D',
    'не хочу на работу :( уже 2 часа ночи...',
]
for example in examples:
    print(f'{example!r}\n  → {clean_text(example)!r}\n')

'RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни :))) ахахаххаха'
  → 'дома ехали на такси пели песни хаха'

'Смотри http://t.co/GqG6iuE2 #пятница, какая ЁЛКА!!! :D'
  → 'смотри какая елка'

'не хочу на работу :( уже 2 часа ночи...'
  → 'не хочу на работу уже часа ночи'



In [8]:
df['clean'] = df['text'].map(clean_text)

print('Скобок после очистки:', df['clean'].str.contains(r'[()]').sum())

Скобок после очистки: 0


### 1.5 Лемматизация

Русский язык богат формами: `люблю`, `любила`, `любим`, `любят` — для TF-IDF это четыре разных признака,
хотя смысл один. Нормализация сводит формы к одной:

- **стемминг** отрезает окончания по правилам (`котиков` → `котик`): быстро, но грубо, `людей` никогда не станет `человек`;
- **лемматизация** приводит слово к словарной форме по морфологическому словарю: `людей` → `человек`, `шёл` → `идти`.

Используется `pymorphy3` — морфологический анализатор русского языка. `parse(word)[0]` - самый вероятный разбор слова, `.normal_form` - его лемма.

Pymorphy возвращает леммы с буквой `ё` (`еще` → `ещё`), поэтому после лемматизации `ё` снова заменяется на `е`.

Ограничение: pymorphy разбирает каждое слово **отдельно, без контекста**, и выбирает самый частый разбор.
Поэтому `начала` (глагол) может стать `начало` (существительное), а `мне` → `я`, `нас` → `мы`.
Смех, сведённый при очистке к `хаха`, превращается в `хах`: главное, что у всех вариантов смеха одна лемма.

В корпусе около 2.7 млн словоупотреблений, но уникальных слов примерно в 15 раз меньше.
Декоратор `@lru_cache` запоминает результат для каждого слова, и повторные слова не разбираются заново.

In [9]:
from functools import lru_cache

import pymorphy3

morph = pymorphy3.MorphAnalyzer()


@lru_cache(maxsize=None)
def lemmatize_word(word: str) -> str:
    return morph.parse(word)[0].normal_form.replace('ё', 'е')


def lemmatize(text: str) -> str:
    return ' '.join(lemmatize_word(word) for word in text.split())


for example in ['любила', 'людей', 'шел', 'лучшего', 'еще', 'не', 'хочу']:
    print(f'{example:>8} → {lemmatize_word(example)}')

  любила → любить
   людей → человек
     шел → идти
 лучшего → хороший
     еще → еще
      не → не
    хочу → хотеть


In [10]:
%%time
df['lemma'] = df['clean'].map(lemmatize)

print('Уникальных слов разобрано:', lemmatize_word.cache_info().currsize)

Уникальных слов разобрано: 160728
CPU times: user 20.4 s, sys: 57.8 ms, total: 20.4 s
Wall time: 20.4 s


In [11]:
df[['text', 'clean', 'lemma']].sample(8, random_state=RANDOM_STATE)

,text,clean,lemma
215081,"“@kasimov_d_n: @tashkentgirl Ты уже начала мне верить? )))”да скоро я буду верить всем,всем кто говорит хотя бы вежл...",ты уже начала мне верить да скоро я буду верить всем всем кто говорит хотя бы вежливо,ты уже начало я верить да скоро я быть верить весь весь кто говорить хотя бы вежливый
220754,RT @JaneAir44660366: о чем можно вообще говорить) http://t.co/3fEZoyWfG0,о чем можно вообще говорить,о чем можно вообще говорить
60463,я ни когда не думал что буду так сильно скучать(((,я ни когда не думал что буду так сильно скучать,я ни когда не думать что быть так сильно скучать
213263,"@EvaKand До меня докопались они на улице по поводу телефона, думаю, от обиды, что я переписал их номера значков и фа...",до меня докопались они на улице по поводу телефона думаю от обиды что я переписал их номера значков и фамилии ну я в...,до я докопаться они на улица по повод телефон думать от обида что я переписать они номер значок и фамилия ну я всегд...
113055,Не надо связываться с моей учительницей по физкультуре:D,не надо связываться с моей учительницей по физкультуре,не надо связываться с мой учительница по физкультура
217274,"ПИТЕРЦЕВ,\r\nВИКА,\r\nСАША..\r\nМНЕ НЕ ХВАТАЕТ ТЕХ РАЗГОВОРОВ :( \r\nЯ СКУЧАЮ ПО ВАМ ((((",питерцев вика саша мне не хватает тех разговоров я скучаю по вам,питерец вика саша я не хватать тот разговор я скучать по вы
12325,@svetik_by ты сёння была ў чорнай шапцы і партфелем? ці я абазнаўся?),ты сення была чорнай шапцы партфелем ц я абазна ся,ты сенний быть чорнать шапец партфель ц я абазный ся
5133,RT @OlyaTaylor: Кто- то громче нас с Аней сегодня смеялся в столовой((((,кто то громче нас с аней сегодня смеялся в столовой,кто то громкий мы с аня сегодня смеяться в столовая


### 1.6 Пустые тексты и дубликаты

После очистки часть твитов становится пустой (в них были только ссылки, упоминания или смайлики),
а часть - одинаковой: ретвиты, «спасибо», «доброе утро» и т.п.

- **Пустые тексты** удаляем: модели нечего по ним предсказывать.
- **Тексты с противоречивыми метками** (один и тот же текст встречается и в позитиве, и в негативе) удаляем все:
  правильного ответа для них нет, это чистый шум разметки.
- **Дубликаты** оставляем по одному. Если одинаковый текст попадёт и в обучающую, и в тестовую выборку,
  модель будет проверяться на том, что уже «видела», и метрики окажутся завышенными.

Дубликаты ищем по колонке `lemma`: именно её увидит модель.

In [12]:
stats = {'исходно': len(df)}

df = df[df['lemma'] != '']
stats['без пустых'] = len(df)

conflicting = df.groupby('lemma')['label'].transform('nunique') > 1
print('Примеры текстов с противоречивыми метками:', df.loc[conflicting, 'lemma'].value_counts().head(5).index.tolist())
df = df[~conflicting]
stats['без противоречивых меток'] = len(df)

df = df.drop_duplicates(subset='lemma').reset_index(drop=True)
stats['без дубликатов'] = len(df)

pd.Series(stats, name='твитов').to_frame()

Примеры текстов с противоречивыми метками: ['хах', 'спасибо', 'да', 'привет', 'спокойный ночь']


,твитов
исходно,226834
без пустых,226717
без противоречивых меток,225982
без дубликатов,210421


In [13]:
df['label'].value_counts(normalize=True).rename(index=LABEL_NAMES).round(3)

label
позитив    0.508
негатив    0.492
Name: proportion, dtype: float64

### 1.7 Проверка: на какие слова можно опираться

Прежде чем обучать модель, посмотрим на простую статистику: в какой доле твитов со словом этот твит позитивный.
Берём слова, которые встречаются хотя бы в 300 твитах. Если бы после очистки осталась утечка
(например, смайлик кириллицей вроде `:ДДД`), здесь всплыли бы «слова» с долей ровно 0 или 1.

In [14]:
from collections import Counter

doc_freq = {label: Counter() for label in LABEL_NAMES}
for text, label in zip(df['lemma'], df['label']):
    doc_freq[label].update(set(text.split()))

word_stats = pd.DataFrame({
    'негатив': pd.Series(doc_freq[0]),
    'позитив': pd.Series(doc_freq[1]),
}).fillna(0).astype(int)
word_stats = word_stats[word_stats.sum(axis=1) >= 300]
word_stats['доля позитива'] = (word_stats['позитив'] / word_stats.sum(axis=1)).round(3)
word_stats = word_stats.sort_values('доля позитива')

pd.concat(
    [word_stats.head(10).reset_index(names='слово'), word_stats.tail(10)[::-1].reset_index(names='слово')],
    axis=1,
    keys=['самые «негативные»', 'самые «позитивные»'],
)

самые «негативные»                               самые «позитивные»          \
               слово негатив позитив доля позитива              слово негатив   
0           печально     386      24         0.059            приятно     104   
1             обидно     762      54         0.066        поздравлять      84   
2            грустно     583      41         0.066                хах    1158   
3             печаль     471      34         0.067            обожать      99   
4             ужасно     770      96         0.111                ура      96   
5           заболеть     715      93         0.115            спасибо     659   
6          сожаление     328      47         0.125               мило      63   
7              жалко     819     132         0.139          спокойный      85   
8            ужасный     583      95         0.140         счастливый     134   
9                увы     271      44         0.140           приятный     103   

                         
  позитив доля позитива  
0     785         0.883  
1     493         0.854  
2    6614         0.851  
3     522         0.841  
4     505         0.840  
5    3429         0.839  
6     317         0.834  
7     384         0.819  
8     589         0.815  
9     450         0.814

**Итоги предобработки:**
- из 226 834 твитов осталось 210 421: удалены пустые после очистки, тексты с противоречивыми метками и дубликаты;
- баланс классов сохранился (50.8% / 49.2%);
- наверху списков осмысленные слова (`печально`, `обидно`, `заболеть` против `приятно`, `поздравлять`, `обожать`),
  и ни одно не встречается только в одном классе: утечки через смайлики не осталось;
- смех (`хах`, ~7.8 тыс. твитов) - один из самых сильных позитивных сигналов. Без нормализации он был бы разбит на сотни написаний.
- около 1% твитов написаны на украинском или белорусском языке. Буквы `і`, `ї`, `є`, `ў` не входят в белый список,
  поэтому слова этих языков после очистки и лемматизации превращаются в редкие токены, которых нет в русском.
  Метки у этих твитов не хуже, чем у остальных (разметка по смайликам не зависит от языка), а редкие токены не попадут в словарь
  благодаря порогу `min_df` в TF-IDF. Отдельный фильтр по языку не применялся.

Для модели используется колонка `lemma`.